In [2]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks.python import vision
from mediapipe.tasks.python import BaseOptions

# Load model
model_path = "hand_landmarker.task"

options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    num_hands=1
)

detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

canvas = None
prev_point = None

# Colors
colors = [(255,0,255), (0,255,0), (0,0,255), (255,255,0)]
color_index = 0
draw_color = colors[color_index]

# Finger detection
def fingers_up(hand):
    return [
        hand[8].y < hand[6].y,   # index
        hand[12].y < hand[10].y, # middle
        hand[16].y < hand[14].y, # ring
        hand[20].y < hand[18].y  # pinky
    ]

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)

    if canvas is None:
        canvas = np.zeros_like(frame)

    # ===== UI Overlay =====
    overlay = frame.copy()
    cv2.rectangle(overlay, (0,0), (640,160), (0,0,0), -1)
    frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

    # Top color palette
    for i, col in enumerate(colors):
        cv2.rectangle(frame, (10+i*70,10), (60+i*70,60), col, -1)

    cv2.putText(frame, "S: Save  C: Clear", (400,50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    # Convert to RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect(mp_image)

    center = None

    if result.hand_landmarks:
        for hand in result.hand_landmarks:

            h, w, _ = frame.shape

            x = int(hand[8].x * w)
            y = int(hand[8].y * h)
            center = (x, y)

            fingers = fingers_up(hand)
            count = fingers.count(True)

            # ===== Gesture Detection =====
            if count == 0:
                gesture = "Fist"
            elif count == 1:
                gesture = "Draw"
            elif count == 2:
                gesture = "Color Change"
            elif count >= 4:
                gesture = "Clear"
            else:
                gesture = "Idle"

            # ===== Display Info =====
            cv2.putText(frame, f'Fingers: {count}', (10,100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

            cv2.putText(frame, f'Gesture: {gesture}', (10,140),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

            # ===== UI Color Selection =====
            if y < 70:
                for i in range(len(colors)):
                    if 10+i*70 < x < 60+i*70:
                        draw_color = colors[i]

            # ===== Gesture Controls =====
            if count == 2:
                color_index = (color_index + 1) % len(colors)
                draw_color = colors[color_index]
                prev_point = None

            if count >= 4:
                canvas = np.zeros_like(frame)

            # ===== Drawing (only index finger) =====
            if fingers[0] and not fingers[1]:
                if prev_point:
                    # smoothing
                    x_s = (prev_point[0] + center[0]) // 2
                    y_s = (prev_point[1] + center[1]) // 2

                    cv2.line(canvas, prev_point, (x_s, y_s),
                             draw_color, 5)

                prev_point = center
            else:
                prev_point = None

            # ===== Fingertip Highlight =====
            cv2.circle(frame, center, 12, draw_color, -1)
            cv2.circle(frame, center, 20, (255,255,255), 2)

    combined = cv2.add(frame, canvas)

    cv2.imshow("🔥 AI Air Canvas Final", combined)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('c'):
        canvas = np.zeros_like(frame)

    if key == ord('s'):
        cv2.imwrite("drawing.png", canvas)

    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()